## 1、如何去构建一个tools

In [11]:
from langchain_core.tools import tool

## 方式1：通过装饰器的方式去定义。在函数的前面上面添加@tool装饰器
@tool
def calculate_expo(base:int,expo:int):
    """
    一个用于计算幂的函数
    :param base:
    :param expo:
    :return:
    """
    return base**expo
type(calculate_expo)

langchain_core.tools.structured.StructuredTool

In [12]:
print(calculate_expo.name)
print(calculate_expo.description)
print(calculate_expo.args)

calculate_expo
一个用于计算幂的函数
:param base:
:param expo:
:return:
{'base': {'title': 'Base', 'type': 'integer'}, 'expo': {'title': 'Expo', 'type': 'integer'}}


In [22]:
## 方式2：通过调用tool函数去定义

def calculate_expo2(base:int,expo:int):
    """
    一个用于计算幂的函数
    :param base:
    :param expo:
    :return:
    """
    return base**expo
calculate_expo2 = tool(calculate_expo2)

# ## 定义函数入参结构时，除了使用typed hint(也就是base:int这种类型推断)外，我们还可以使用我们的pydantic
# from pydantic import BaseModel,Field
# def calculate_expo3(base,expo):
#     return base**expo
# class CalcExpoSchema(BaseModel):
#     base:int = Field(description="幂的底数")
#     expo:int = Field(description="幂的指数")
# calculate_expo3 = tool(calculate_expo3,args_schema=CalcExpoSchema,description="一个用于计算幂的函数")
# calculate_expo3.args

# print(type(calculate_expo2))
# print(type(a_new_tool))
args={"base":2,"expo":3}
globals()['calculate_expo2'].invoke(args)

8

## 2、如何去调用tool
通过调用invoke去执行tool当中的逻辑，invoke传入一个dict，dict的键就是tool的入参，值就是参数的值

In [27]:
calculate_expo3.invoke({"base":"2","expo":3})

8

## 3、大模型如何感知到有哪些tools

In [1]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

C:\Users\m1881\miniconda3\envs\LangChainProj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import logging
logging.basicConfig(level=logging.DEBUG,format="%(asctime)s %(filename)s %(message)s")

In [3]:
llm.invoke("你好")

2025-11-10 09:55:38,639 connectionpool.py Starting new HTTPS connection (1): api.smith.langchain.com:443
2025-11-10 09:55:38,667 _base_client.py Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-c3f378d9-6208-4423-9bb0-0f2064f3c1a4', 'json_data': {'messages': [{'content': '你好', 'role': 'user'}], 'model': 'gpt-4o-mini', 'stream': False}}
2025-11-10 09:55:38,668 _base_client.py Sending HTTP Request: POST https://api.openai-proxy.org/v1/chat/completions
2025-11-10 09:55:38,669 _trace.py connect_tcp.started host='127.0.0.1' port=7897 local_address=None timeout=None socket_options=None
2025-11-10 09:55:38,669 _trace.py connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x00000174C404F920>
2025-11-10 09:55:38,669 _trace.py send_request_headers.started request=<Request [b'CONNECT']>
2025-11-10 09:55:38,670 _trace.py send_request_headers.complete

AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaBQ33OiEvug0Dnc3l7SgQSZKduDE', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--238abecc-1d24-42cd-a2c7-e3f9255e1c47-0', usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

2025-11-10 09:55:40,092 client.py Sending compressed multipart request with context: trace=238abecc-1d24-42cd-a2c7-e3f9255e1c47,id=238abecc-1d24-42cd-a2c7-e3f9255e1c47
2025-11-10 09:55:40,379 connectionpool.py https://api.smith.langchain.com:443 "POST /runs/multipart HTTP/1.1" 202 34


In [8]:
tools=[calculate_expo3]
llm_with_tools_bound = llm.bind_tools(tools)

In [9]:
res = llm_with_tools_bound.invoke("帮我计算一下2的10次方是多少")

2025-11-10 09:57:17,282 _base_client.py Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-dd0a99fc-5ae0-4635-b9d3-23d152a176c1', 'json_data': {'messages': [{'content': '帮我计算一下2的10次方是多少', 'role': 'user'}], 'model': 'gpt-4o-mini', 'stream': False, 'tools': [{'type': 'function', 'function': {'name': 'calculate_expo3', 'description': '一个用于计算幂的函数', 'parameters': {'properties': {'base': {'description': '幂的底数', 'type': 'integer'}, 'expo': {'description': '幂的指数', 'type': 'integer'}}, 'required': ['base', 'expo'], 'type': 'object'}}}]}}
2025-11-10 09:57:17,284 _base_client.py Sending HTTP Request: POST https://api.openai-proxy.org/v1/chat/completions
2025-11-10 09:57:17,284 _trace.py connect_tcp.started host='127.0.0.1' port=7897 local_address=None timeout=None socket_options=None
2025-11-10 09:57:17,285 _trace.py connect_tcp.complete return_value=<httpcore._backends.sync.Syn

In [30]:
res # 此时AIMessage当中的content没有内容。当前AI无法自动去调用工具，需要我们手动去调用

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 73, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaBJkp8U4ZbtTamTbrnlJPdIpuet1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--0378cd4e-0ae6-43ba-bc11-5b0ef730de39-0', tool_calls=[{'name': 'calculate_expo3', 'args': {'base': 2, 'expo': 10}, 'id': 'call_yo6ym2OTAT3rzJGxgeEKm7hM', 'type': 'tool_call'}], usage_metadata={'input_tokens': 73, 'output_tokens': 21, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:
# str2tool={
#     "calculate_expo3":pydantic_tool,
#     # 维护其他tool_name和真正的tool实例之间的映射
# }
for tool_call in res.tool_calls:
    print(tool_call)
    tool_name = tool_call['name']
    args = tool_call['args']
    tool_invoke_res = globals()[tool_name](base=2,expo=10)
    print(tool_invoke_res)


{'name': 'calculate_expo3', 'args': {'base': 2, 'expo': 10}, 'id': 'call_yo6ym2OTAT3rzJGxgeEKm7hM', 'type': 'tool_call'}
1024


In [10]:
globals()["calc_expo"]

KeyError: 'calc_expo'

## 4、tools的底层原理
利用大模型厂商所提供的工具调用的能力。